# Triage Copilot — app (dashboard)

Loads the trained bundles from `models/` and serves them as a single-page dashboard (sidebar, Case details + Triage prediction panels, full-width Similar cases). **Trains nothing.** The model exploration view is unlocked via **Settings → Admin mode**, which then reveals the 'View models' button.

## 1. Load the trained bundles

In [ ]:
import os, sys, warnings, html
warnings.filterwarnings("ignore")
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"
import numpy as np, pandas as pd, joblib

search = os.getcwd()
while search != os.path.dirname(search) and not os.path.exists(os.path.join(search, "case_pipeline.py")):
    search = os.path.dirname(search)
sys.path.insert(0, search)
import case_pipeline as P

def _load(key):
    path = os.path.join(P.MODELS_DIR, f"{key}.joblib")
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} missing — run training/train_{key}.ipynb first")
    return joblib.load(path)

CLASSIFIERS   = ["m1_case_type", "m2_category", "m3_priority_direct",
                 "m3_priority_review", "m4_escalation", "m5_route"]
EXPLORE_ORDER = ["m1_case_type", "m2_category", "m3_priority_direct",
                 "m3_priority_review", "m5_route", "m4_escalation"]
MODEL     = {key: _load(key) for key in CLASSIFIERS}
RETRIEVER = _load("similar_cases")["retriever"]   # TF-IDF + nearest-neighbour index
TAGGER    = _load("tag_suggester")                # tag suggestions (never a model input)
import random
_full_df = P.load_clean()
_, HELDOUT_POOL = P.group_split(_full_df)   # real cases the text models never trained on
print("loaded", len(MODEL), "models + similar-case search + tag suggester")

## 2. Prediction cascade

Case type and category are predicted from the summary first, then feed the priority and escalation models. Tags are never an input (they are leakage).

In [ ]:
PRIORITIES = MODEL["m3_priority_review"]["meta"]["priorities"]
PRIORITY_RANK = {level: i for i, level in enumerate(PRIORITIES)}

def _frame(base, extra=None):
    return pd.DataFrame([{**base, **(extra or {})}])

def _ranked(key, row):
    bundle = MODEL[key]; model = bundle["model"]
    proba = model.predict_proba(row[bundle["features"]])[0]
    order = proba.argsort()[::-1]
    return [(str(model.classes_[i]), float(proba[i])) for i in order]

def predict_case(summary, channel, plan, month, tenure, agent_priority=None):
    base = {"case_summary": summary or "", "channel": channel, "plan_tier": plan,
            "month_num": int(month), "customer_tenure_months": int(tenure)}
    case_type = _ranked("m1_case_type", _frame(base))[0]
    category  = _ranked("m2_category",  _frame(base))[0]
    priority_direct = _ranked("m3_priority_direct", _frame(base))[0]
    priority_review = _ranked("m3_priority_review", _frame(base, {"case_type": case_type[0]}))[0]
    route = _ranked("m5_route", _frame(base))[0]
    esc_bundle = MODEL["m4_escalation"]; esc_model = esc_bundle["model"]
    esc_proba = esc_model.predict_proba(_frame(base, {"category": category[0]})[esc_bundle["features"]])[0]
    esc_prob = float(esc_proba[list(esc_model.classes_).index(True)])
    result = {"case_type": case_type, "category": category, "route": route,
              "priority_direct": priority_direct, "priority_review": priority_review,
              "priorities_agree": priority_direct[0] == priority_review[0],
              "esc_prob": esc_prob, "esc_flag": esc_prob >= esc_bundle["threshold"],
              "needs_review": route[1] < MODEL["m5_route"].get("review_threshold", 0.0),
              "terms": _top_terms("m1_case_type", _frame(base), 3)}
    if agent_priority in PRIORITY_RANK:
        gap = abs(PRIORITY_RANK[agent_priority] - PRIORITY_RANK[priority_review[0]])
        result["agent_priority"] = agent_priority
        result["priority_flag"] = gap >= MODEL["m3_priority_review"]["meta"]["flag_gap"]
    return result

_STOPWORDS = set("the a an and or of to in for with on at is are be was were this that it its my our "
                 "your i we you they he she as by from not no cannot can keeps after getting get got need "
                 "want please thanks cheers hi hello there their them has have had do does did been being but "
                 "so if then when which who what why how able keep still just also very more most".split())

def _top_terms(key, row, k=3):
    bundle = MODEL[key]; model = bundle["model"]
    features = model.named_steps["features"]; clf = model.named_steps["model"]
    names = features.get_feature_names_out()
    weights = np.asarray(features.transform(row[bundle["features"]]).todense()).ravel()
    class_idx = list(clf.classes_).index(model.predict(row[bundle["features"]])[0])
    coef = clf.coef_
    coef_row = coef[class_idx] if coef.shape[0] > 1 else (coef[0] if class_idx == 1 else -coef[0])
    contribution = coef_row * weights
    prefix = "case_summary_w__"
    candidates = sorted((i for i in range(len(names)) if names[i].startswith(prefix)),
                        key=lambda i: contribution[i], reverse=True)
    terms = []
    for i in candidates:
        if contribution[i] <= 0: break
        term = " ".join(w for w in names[i].replace(prefix, "").split() if w not in _STOPWORDS and len(w) > 2)
        if term and term not in terms: terms.append(term)
        if len(terms) >= k: break
    return terms

## 3. Dashboard renderers (prediction panel, similar-cases table)

In [ ]:
# Small inline SVG icons (18px, stroke = currentColor).
def _svg(body): return f'<svg viewBox="0 0 24 24" width="18" height="18" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round">{body}</svg>'
ICONS = {
    "doc":   _svg('<path d="M14 2H6a2 2 0 0 0-2 2v16a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2V8z"/><path d="M14 2v6h6"/>'),
    "tag":   _svg('<path d="M20.59 13.41 13.42 20.6a2 2 0 0 1-2.83 0L2 12V2h10l8.59 8.59a2 2 0 0 1 0 2.82z"/><circle cx="7" cy="7" r="1.5"/>'),
    "flag":  _svg('<path d="M4 15s1-1 4-1 5 2 8 2 4-1 4-1V4s-1 1-4 1-5-2-8-2-4 1-4 1z"/><line x1="4" y1="22" x2="4" y2="15"/>'),
    "eye":   _svg('<path d="M2 12s3.5-7 10-7 10 7 10 7-3.5 7-10 7-10-7-10-7z"/><circle cx="12" cy="12" r="3"/>'),
    "route": _svg('<circle cx="6" cy="19" r="2"/><circle cx="18" cy="5" r="2"/><path d="M12 19h4a2 2 0 0 0 2-2V8M8 5H6a2 2 0 0 0-2 2v10"/>'),
    "alert": _svg('<circle cx="12" cy="12" r="10"/><line x1="12" y1="8" x2="12" y2="12"/><line x1="12" y1="16" x2="12.01" y2="16"/>'),
    "bulb":  _svg('<path d="M9 18h6M10 22h4M12 2a7 7 0 0 0-4 12.7c.6.5 1 1.3 1 2.3h6c0-1 .4-1.8 1-2.3A7 7 0 0 0 12 2z"/>'),
    "spark": _svg('<path d="M12 3l1.9 5.1L19 10l-5.1 1.9L12 17l-1.9-5.1L5 10l5.1-1.9z"/>'),
    "book":  _svg('<path d="M4 19.5A2.5 2.5 0 0 1 6.5 17H20"/><path d="M6.5 2H20v20H6.5A2.5 2.5 0 0 1 4 19.5v-15A2.5 2.5 0 0 1 6.5 2z"/>'),
    "moon":  _svg('<path d="M21 12.8A9 9 0 1 1 11.2 3 7 7 0 0 0 21 12.8z"/>'),
    "hist":  _svg('<path d="M3 3v5h5"/><path d="M3.05 13A9 9 0 1 0 6 5.3L3 8"/><path d="M12 7v5l4 2"/>'),
    "gear":  _svg('<circle cx="12" cy="12" r="3"/><path d="M19.4 15a1.65 1.65 0 0 0 .33 1.82l.06.06a2 2 0 1 1-2.83 2.83l-.06-.06a1.65 1.65 0 0 0-1.82-.33 1.65 1.65 0 0 0-1 1.51V21a2 2 0 0 1-4 0v-.09A1.65 1.65 0 0 0 9 19.4a1.65 1.65 0 0 0-1.82.33l-.06.06a2 2 0 1 1-2.83-2.83l.06-.06a1.65 1.65 0 0 0 .33-1.82 1.65 1.65 0 0 0-1.51-1H3a2 2 0 0 1 0-4h.09A1.65 1.65 0 0 0 4.6 9a1.65 1.65 0 0 0-.33-1.82l-.06-.06a2 2 0 1 1 2.83-2.83l.06.06a1.65 1.65 0 0 0 1.82.33H9a1.65 1.65 0 0 0 1-1.51V3a2 2 0 0 1 4 0v.09a1.65 1.65 0 0 0 1 1.51 1.65 1.65 0 0 0 1.82-.33l.06-.06a2 2 0 1 1 2.83 2.83l-.06.06a1.65 1.65 0 0 0-.33 1.82V9a1.65 1.65 0 0 0 1.51 1H21a2 2 0 0 1 0 4h-.09a1.65 1.65 0 0 0-1.51 1z"/>'),
    "check": _svg('<path d="M20 6 9 17l-5-5"/>'),
    "target":_svg('<circle cx="12" cy="12" r="10"/><circle cx="12" cy="12" r="6"/><circle cx="12" cy="12" r="2"/>'),
}

def _risk_band(prob):
    thr = MODEL["m4_escalation"]["threshold"]
    if prob >= thr:     return "High",   "#dc2626", "#fef2f2"
    if prob >= thr / 2: return "Medium", "#d97706", "#fffbeb"
    return "Low", "#059669", "#ecfdf5"

def _why(r):
    parts = []
    if r["terms"]:
        parts.append(", ".join(r["terms"]) + " in summary")
    parts.append(f'{r["case_type"][0].replace("_", " ")} language')
    if r["esc_flag"]:
        parts.append("flagged above the escalation threshold")
    elif r["esc_prob"] >= MODEL["m4_escalation"]["threshold"] / 2:
        parts.append("elevated but below the escalation threshold")
    else:
        parts.append("below the escalation threshold")
    return " + ".join(parts)

def _bar(pct, colour):
    return (f'<span class="confwrap"><span class="pct">{pct:.0f}%</span>'
            f'<span class="track"><span class="fill" style="width:{pct:.0f}%;background:{colour}"></span></span></span>')

def _prow(icon, bg, fg, label, value, conf="", wide=False):
    vclass = "pvalue pvwide" if wide else "pvalue"
    return (f'<div class="prow"><span class="picon" style="background:{bg};color:{fg}">{ICONS[icon]}</span>'
            f'<span class="plabel">{label}</span><span class="{vclass}">{value}</span>'
            f'<span class="pconf">{conf}</span></div>')

def render_prediction(r):
    rows = [
        _prow("doc",   "#eef2ff", "#6366f1", "Case type",        r["case_type"][0].replace("_"," ").title(), _bar(r["case_type"][1]*100, "#6366f1")),
        _prow("tag",   "#ecfdf5", "#10b981", "Category",         r["category"][0].replace("_"," ").title(),  _bar(r["category"][1]*100, "#3b82f6")),
        _prow("flag",  "#fef2f2", "#ef4444", "Priority · Direct", r["priority_direct"][0],               _bar(r["priority_direct"][1]*100, "#f59e0b")),
        _prow("eye",   "#fffbeb", "#f59e0b", "Priority · Review", r["priority_review"][0],               _bar(r["priority_review"][1]*100, "#f59e0b")),
        _prow("route", "#ecfeff", "#06b6d4", "Suggested route",  r["route"][0].title(), ""),
    ]
    level, colour, bg = _risk_band(r["esc_prob"])
    pill = (f'<span class="riskpill" style="color:{colour};background:{bg}">'
            f'<span class="dot" style="background:{colour}"></span>{level} escalation risk · {r["esc_prob"]*100:.0f}%</span>')
    rows.append(_prow("alert", "#fef2f2", "#ef4444", "Risk", pill, wide=True))
    rows.append(_prow("bulb",  "#fffbeb", "#f59e0b", "Why", f'<span class="why">{html.escape(_why(r))}</span>', wide=True))
    if r.get("agent_priority"):
        if r.get("priority_flag"):
            msg = (f'Agent set <b>{r["agent_priority"]}</b>, model estimates '
                   f'<b>{r["priority_review"][0]}</b>. Please review')
            rows.append(_prow("alert", "#fef2f2", "#ef4444", "Priority check", f'<span class="why">{msg}</span>', wide=True))
        else:
            msg = f'Agent priority <b>{r["agent_priority"]}</b> looks consistent'
            rows.append(_prow("check", "#ecfdf5", "#059669", "Priority check", f'<span class="why">{msg}</span>', wide=True))
    return '<div class="predbody">' + "".join(rows) + "</div>"

def _risk_badge(escalated, priority):
    if escalated:                          lvl, c, bg = "High",   "#dc2626", "#fef2f2"
    elif priority in ("High", "Urgent"):   lvl, c, bg = "Medium", "#d97706", "#fffbeb"
    else:                                  lvl, c, bg = "Low",    "#059669", "#ecfdf5"
    return f'<span class="badge" style="color:{c};background:{bg}"><span class="dot" style="background:{c}"></span>{lvl}</span>'

def render_similar(summary, min_sim=0.5, max_rows=25):
    """Show every past case scoring at least `min_sim` cosine similarity (no fixed count)."""
    sim = P.query_similar(RETRIEVER, summary, k=50)
    sim = sim[sim["similarity"] >= min_sim].head(max_rows)
    if sim.empty:
        return '<div class="empty">No past cases above 50% similarity for this summary.</div>'
    body = ""
    for _, c in sim.iterrows():
        created = pd.to_datetime(c["created_at"]).strftime("%b %Y") if pd.notna(c["created_at"]) else "—"
        pct = c["similarity"] * 100
        body += ("<tr>"
                 f'<td class="t-title">{html.escape(str(c["case_summary"])[:46])}</td>'
                 f'<td>{str(c["case_type"]).replace("_"," ").title()}</td>'
                 f'<td>{str(c["category"]).replace("_"," ").title()}</td>'
                 f'<td>{c["priority"]}</td>'
                 f'<td>{str(c["assigned_team"]).title()}</td>'
                 f'<td>{_risk_badge(bool(c["escalated"]), c["priority"])}</td>'
                 f'<td><span class="simwrap"><span class="simtrack"><span class="simfill" style="width:{pct:.0f}%"></span></span>'
                 f'<span class="simpct">{pct:.0f}%</span></span></td>'
                 f'<td class="muted">{created}</td></tr>')
    head = ("<tr><th>Title</th><th>Case type</th><th>Category</th><th>Priority</th>"
            "<th>Route</th><th>Risk</th><th>Similarity</th><th>Created</th></tr>")
    return f'<div class="simscroll"><table class="simtable">{head}{body}</table></div>'

print("renderers ready")

## 4. Exploration-view helpers (reached via 'View models')

In [ ]:
def model_cards():
    cards = ""
    for key in EXPLORE_ORDER:
        b = MODEL[key]; meta = b["meta"]; m = b["metrics"]
        core = ["accuracy", "macro_F1", "weighted_F1", "cohen_kappa", "ROC_AUC", "PR_AUC"]
        core_str = " · ".join(f'{c} <b>{m[c]}</b>' for c in core if c in m)
        extra = [k for k in m if k not in core + ["n_test", "baseline_acc"]]
        extra_str = " · ".join(f'{k} <b>{m[k]}</b>' for k in extra)
        cards += (f'<div class="mcard"><div class="mh">{key.split("_")[0].upper()} — {b["label"]}'
                  f'<span class="chip">{meta.get("contributor","")}</span></div><div class="mb">'
                  f'<div class="spec"><span>Approach</span>{meta["approach"]}</div>'
                  f'<div class="spec"><span>Inputs</span>{meta["inputs"]}</div>'
                  f'<div class="spec"><span>Split</span>{meta["split"]}</div>'
                  f'<div class="spec"><span>Core</span>{core_str}</div>'
                  + (f'<div class="spec"><span>Signature</span>{extra_str}</div>' if extra_str else "")
                  + (f'<div class="cav">{meta.get("note","")}</div>' if meta.get("note") else "")
                  + "</div></div>")
    return f'<div class="mgrid">{cards}</div>'

def metrics_table():
    rows = []
    for key in EXPLORE_ORDER:
        b = MODEL[key]; row = {"Model": key.split("_")[0].upper(), "Predicts": b["label"]}
        row.update(b["metrics"]); rows.append(row)
    cols = ["Model","Predicts","n_test","accuracy","macro_F1","weighted_F1","cohen_kappa","ROC_AUC","PR_AUC","baseline_PR_AUC","baseline_acc"]
    frame = pd.DataFrame(rows)
    return frame[[c for c in cols if c in frame.columns]]

def case_choices(key):
    test = MODEL[key]["test"]
    return [(f'{r.case_id} — {str(r.case_summary)[:55]}', i) for i, r in test.iterrows()]

def inspect(key, idx):
    if key is None or idx is None:
        return '<div class="rescard"><p class="note">Pick a model and a case.</p></div>'
    b = MODEL[key]; row = b["test"].iloc[int(idx)]
    actual, pred = row[b["target"]], row["predicted"]; conf = row.get("confidence")
    ok = str(pred) == str(actual)
    mark = '<span class="tag ok">match</span>' if ok else '<span class="tag warn">differs</span>'
    extra = f' · prob {row["prob"]:.0%} vs threshold {b.get("threshold",0):.0%}' if b["kind"] == "binary" else ""
    return (f'<div class="rescard"><p class="sum">“{html.escape(str(row.case_summary)[:300])}”</p>'
            f'<table class="insp"><tr><td class="k">Model</td><td>{key.split("_")[0].upper()} — {b["label"]}</td></tr>'
            f'<tr><td class="k">Predicted</td><td>{html.escape(str(pred))}' + (f' ({conf:.0%})' if conf is not None else "") + f'{extra}</td></tr>'
            f'<tr><td class="k">Actual</td><td>{html.escape(str(actual))}</td></tr>'
            f'<tr><td class="k">Result</td><td>{mark}</td></tr></table>'
            f'<p class="note">Held-out case ({b["meta"]["split"]} split) — never seen in training.</p></div>')

def confusion_html(key):
    cm = MODEL[key]["confusion"]; labels = cm["labels"]; matrix = cm["matrix"]
    peak = max((max(r) for r in matrix), default=1) or 1
    head = "".join(f'<th>{html.escape(str(l))[:10]}</th>' for l in labels)
    body = ""
    for i, label in enumerate(labels):
        tds = ""
        for j in range(len(labels)):
            v = matrix[i][j]; shade = 0.08 + 0.72 * (v / peak)
            tds += f'<td style="background:rgba(99,102,241,{shade:.2f})">{v}</td>' if v else f'<td>{v}</td>'
        body += f'<tr><th>{html.escape(str(label))[:10]}</th>{tds}</tr>'
    return (f'<div class="rescard"><div class="small muted">rows = actual · columns = predicted</div>'
            f'<table class="cm"><tr><th></th>{head}</tr>{body}</table></div>')

def report_html(key):
    report = MODEL[key]["report"]; rows = ""
    for cls, d in report.items():
        if cls in ("accuracy", "macro avg", "weighted avg") or not isinstance(d, dict): continue
        rows += (f'<tr><td>{html.escape(str(cls))}</td><td>{d["precision"]:.2f}</td>'
                 f'<td>{d["recall"]:.2f}</td><td>{d["f1-score"]:.2f}</td><td>{int(d["support"])}</td></tr>')
    return f'<div class="rescard"><table class="rep"><tr><th>class</th><th>prec</th><th>rec</th><th>F1</th><th>n</th></tr>{rows}</table></div>'


def heldout_table(key, sort="Case order", limit=500):
    """Browsable table of held-out cases for a model. `sort` orders the Result column:
    'Case order' (default), 'Correct first', or 'Wrong first'."""
    b = MODEL[key]; t = b["test"].head(limit)
    ok_series = t["predicted"].astype(str) == t[b["target"]].astype(str)
    records = list(zip(list(t.iterrows()), list(ok_series)))   # ((idx, row), ok)
    if sort == "Correct first":
        records.sort(key=lambda x: not x[1])   # correct (True) first; stable within group
    elif sort == "Wrong first":
        records.sort(key=lambda x: x[1])       # wrong (False) first; stable within group
    rows = ""
    for (_, r), ok in records:
        conf = r.get("confidence")
        conf_str = f'{conf:.0%}' if conf is not None and pd.notna(conf) else "—"
        mark = '<span class="tag ok">correct</span>' if ok else '<span class="tag warn">wrong</span>'
        rows += (f'<tr><td class="muted">{html.escape(str(r["case_id"]))}</td>'
                 f'<td class="t-title">{html.escape(str(r["case_summary"])[:72])}</td>'
                 f'<td>{html.escape(str(r["predicted"]))}</td><td>{conf_str}</td>'
                 f'<td>{html.escape(str(r[b["target"]]))}</td><td>{mark}</td></tr>')
    head = ('<tr><th>Case</th><th>Summary</th><th>Predicted</th><th>Conf</th>'
            '<th>Actual</th><th>Result</th></tr>')
    cap = (f'<div class="small muted">{len(t)} held-out cases · {ok_series.mean():.0%} correct · '
           f'{b["meta"]["split"]} split (never seen in training)</div>')
    return (f'<div class="rescard">{cap}<div class="simscroll" style="max-height:380px;overflow:auto">'
            f'<table class="simtable">{head}{rows}</table></div></div>')
print("exploration helpers ready |", metrics_table().shape)

## 5. Dashboard UI

In [ ]:
import gradio as gr

CHANNELS = ["email", "webchat", "in_app", "phone"]
PLANS    = ["Free", "Standard", "Pro", "Enterprise"]
EXPLORE_LABELS = {k: f'{k.split("_")[0].upper()} — {MODEL[k]["label"]}' for k in EXPLORE_ORDER}
DOWNLOAD_KEYS = EXPLORE_ORDER + ["similar_cases", "tag_suggester"]
NOTEBOOK_PATH  = {k: os.path.join(P.ROOT, "training", f"train_{k}.ipynb") for k in DOWNLOAD_KEYS}

# Worked example used for the initial view and the "Load example" action.
EX_SUMMARY = "Cannot connect the Xero integration — the OAuth token keeps failing with a 500 error after the latest update"
EXAMPLE = [EX_SUMMARY, "in_app", "Enterprise", 6, 30, "Medium", "xero;integration;error"]

def triage(summary, channel, plan, month, tenure, agent_priority, tags):
    """Run the cascade, render the panels, and auto-suggest tags if the field is empty.
    Tags are a workflow aid only — they are never fed back into the models."""
    if not (summary or "").strip():
        return EMPTY_PRED, EMPTY_SIM, gr.update()
    result = predict_case(summary, channel, plan, month, tenure,
                          None if agent_priority in (None, "(none)") else agent_priority)
    prediction = render_prediction(result)
    similar = render_similar(summary)
    if not (tags or "").strip():
        suggested = ";".join(t for t, _ in P.suggest_tags(TAGGER, summary, k=5))
        tags_out = gr.update(value=suggested)
    else:
        tags_out = gr.update()
    return prediction, similar, tags_out

def counter(summary):
    return f'<div class="counter">{len(summary or "")}/500</div>'

# The dashboard opens empty; the user types a summary (or, in admin mode, loads an example).
EMPTY_PRED = '<div class="empty">Enter a case summary and press <b>Run triage</b>.</div>'
EMPTY_SIM  = '<div class="empty">Similar past cases appear here after you run triage.</div>'
INIT_PRED, INIT_SIM, INIT_COUNT = EMPTY_PRED, EMPTY_SIM, counter("")

def pick_example():
    """Load example pulls a real held-out case (unseen in training), varying each click."""
    r = HELDOUT_POOL.sample(1).iloc[0]
    return [str(r["case_summary"]), str(r["channel"]), str(r["plan_tier"]),
            int(r["month_num"]), int(r["customer_tenure_months"]), "(none)", ""]

def clear_all():
    """Reset the whole case interface to its empty state."""
    return ["", "email", "Standard", 6, 12, "(none)", "", EMPTY_PRED, EMPTY_SIM, counter("")]

# Sidebar is split so the nav items can be real (clickable) Gradio buttons.
BRAND_HTML   = f'<div class="brand">{ICONS["spark"]}<span>Northstar</span></div>'
PROFILE_HTML = '<div class="profile"><span class="avatar">JD</span><span class="pname">John Doe</span></div>'

HEADER = f"""
<div class="hdr">
  <div class="hdr-icon">{ICONS['spark']}</div>
  <div><div class="hdr-title">Triage Copilot</div>
  <div class="hdr-sub">Classify, route, prioritise and flag risk for a new case.</div></div>
</div>"""

CASE_HEAD = f"""<div class="panel-head"><span class="ph-icon" style="background:#eef2ff;color:#6366f1">{ICONS['doc']}</span>
  <div><div class="ph-title">Case details</div><div class="ph-sub">Provide the essential information to triage a new case.</div></div>
  <span class="area-tag area-in">Input</span></div>"""

PRED_HEAD = f"""<div class="panel-head"><span class="ph-icon" style="background:#ecfdf5;color:#10b981">{ICONS['target']}</span>
  <div><div class="ph-title">Triage prediction</div><div class="ph-sub">Model results for this case.</div></div>
  <span class="area-tag area-out">Output</span></div>"""

SIM_HEAD = f"""<div class="panel-head"><span class="ph-icon" style="background:#ecfeff;color:#06b6d4">{ICONS['route']}</span>
  <div><div class="ph-title">Similar cases</div><div class="ph-sub">Based on past cases with similar issues.</div></div>
  <span class="simlink">≥ 50% similarity</span></div>"""

SETTINGS_HEAD = f"""<div class="panel-head"><span class="ph-icon" style="background:#eef2ff;color:#6366f1">{ICONS['gear']}</span>
  <div><div class="ph-title">Settings</div><div class="ph-sub">Workspace preferences and access.</div></div></div>"""

CSS = r"""
:root{--bg:#f3f4f8;--card:#fff;--ink:#0f172a;--muted:#64748b;--line:#e8eaf0;--accent:#6d5efc;}
.gradio-container{background:var(--bg)!important;max-width:100%!important;padding:0!important;}
footer{display:none!important;}
#shell{gap:0!important;flex-wrap:nowrap!important;align-items:stretch!important;}
/* sidebar */
#sidebar{flex:0 0 212px!important;min-width:212px!important;display:flex!important;flex-direction:column!important;}
.sb{background:#fff;border-right:1px solid var(--line);min-height:100vh;padding:20px 14px;display:flex;flex-direction:column;}
.brand{display:flex;align-items:center;gap:9px;font-weight:800;font-size:19px;color:var(--ink);padding:4px 8px 18px;}
.brand svg{color:var(--accent);}
#nav-triage button,#nav-settings button{width:100%!important;justify-content:flex-start!important;text-align:left!important;background:#fff!important;border:none!important;box-shadow:none!important;color:#475569!important;font-weight:600!important;font-size:14px!important;padding:10px 12px!important;border-radius:10px!important;min-height:unset!important;}
#nav-triage button:hover,#nav-settings button:hover{background:#f1f2f9!important;}
#nav-triage button{background:#eef0ff!important;color:var(--accent)!important;}
#profile-wrap{margin-top:auto;}
.profile{display:flex;align-items:center;gap:9px;padding:10px 8px;border-top:1px solid var(--line);}
.avatar{width:30px;height:30px;border-radius:50%;background:var(--accent);color:#fff;display:flex;align-items:center;justify-content:center;font-size:12px;font-weight:700;}
.pname{font-weight:600;font-size:13px;color:var(--ink);}
/* content */
#content{padding:22px 28px 40px!important;}
.hdr{display:flex;align-items:center;gap:12px;}
.hdr-icon{color:var(--accent);} .hdr-icon svg{width:26px;height:26px;}
.hdr-title{font-size:25px;font-weight:800;color:var(--ink);letter-spacing:-.4px;}
.hdr-sub{color:var(--muted);font-size:14px;margin-top:2px;}
#topbar{align-items:center!important;margin-bottom:18px;gap:10px!important;}
#topbar-left{flex:1!important;}
#view-btn{flex:0 0 auto!important;max-width:170px;}
#view-btn button{background:#fff!important;color:var(--accent)!important;border:1px solid #d9dbf7!important;border-radius:10px!important;font-weight:700!important;box-shadow:none!important;}
/* main-tab area distinction + equal-height top row */
#maintop{align-items:stretch!important;gap:18px!important;}
#panel-input,#panel-output{height:100%!important;display:flex!important;flex-direction:column!important;}
#panel-input{border-top:3px solid #6d5efc!important;}
#panel-output{border-top:3px solid #10b981!important;}
#panel-sim{border-top:3px solid #06b6d4!important;margin-top:18px!important;}
.area-tag{margin-left:auto;font-weight:700;font-size:12px;padding:4px 11px;border-radius:999px;}
.area-in{color:var(--accent);background:#eef0ff;} .area-out{color:#059669;background:#ecfdf5;}
/* cards / panels */
.panel{background:var(--card)!important;border:1px solid var(--line)!important;border-radius:18px!important;padding:20px 22px!important;box-shadow:0 1px 2px rgba(15,23,42,.04)!important;}
.panel-head{display:flex;align-items:center;gap:12px;margin-bottom:16px;}
.ph-icon{width:40px;height:40px;border-radius:12px;display:flex;align-items:center;justify-content:center;flex:0 0 40px;}
.ph-title{font-size:18px;font-weight:800;color:var(--ink);}
.ph-sub{font-size:13px;color:var(--muted);margin-top:1px;}
.ready{margin-left:auto;display:inline-flex;align-items:center;gap:5px;background:#ecfdf5;color:#059669;font-weight:700;font-size:13px;padding:5px 11px;border-radius:999px;}
.ready svg{width:15px;height:15px;}
.simlink{margin-left:auto;color:#06b6d4;font-weight:700;font-size:13px;background:#ecfeff;padding:6px 12px;border-radius:999px;}
/* case-details fields */
.field-head{font-weight:700;color:var(--ink);font-size:15px;} .field-head .req{color:#ef4444;}
.field-sub{color:var(--muted);font-size:13px;margin:-2px 0 10px;}
.counter{text-align:right;color:#94a3b8;font-size:12px;margin-top:-4px;}
#loadex{flex:0 0 auto!important;}
#loadex button{background:#eef2ff!important;color:var(--accent)!important;border:none!important;font-weight:700!important;border-radius:9px!important;box-shadow:none!important;font-size:13px!important;padding:6px 12px!important;}
#clear-top{flex:0 0 auto!important;}
#clear-top button{background:#fff!important;border:1px solid var(--line)!important;color:#475569!important;font-weight:700!important;border-radius:9px!important;box-shadow:none!important;font-size:13px!important;padding:6px 14px!important;min-height:unset!important;}
#runbtn button{background:linear-gradient(90deg,#6d5efc,#5b7cfa)!important;color:#fff!important;border:none!important;border-radius:12px!important;font-weight:800!important;font-size:15px!important;padding:13px!important;box-shadow:0 4px 14px rgba(109,94,252,.3)!important;}
.panel .gr-textbox textarea,.panel textarea{border-radius:12px!important;border:1px solid var(--line)!important;font-size:15px!important;background:#fcfcfe!important;}
.panel input,.panel select,.panel .wrap{border-radius:11px!important;}
/* prediction rows */
.predbody{display:flex;flex-direction:column;}
.prow{display:grid;grid-template-columns:40px 128px 1fr 168px;align-items:center;gap:10px;padding:13px 2px;border-top:1px solid #f1f2f7;}
.prow:first-child{border-top:none;}
.picon{width:34px;height:34px;border-radius:10px;display:flex;align-items:center;justify-content:center;}
.picon svg{width:17px;height:17px;}
.plabel{color:var(--muted);font-size:14px;font-weight:600;}
.pvalue{font-weight:800;color:var(--ink);font-size:16px;} .pvwide{grid-column:3 / -1;}
.pvalue .why{font-weight:500;font-size:14px;color:#334155;line-height:1.5;}
.confwrap{display:flex;flex-direction:column;align-items:flex-end;gap:5px;}
.pct{font-size:13px;font-weight:800;color:var(--accent);}
.track{width:150px;height:7px;border-radius:999px;background:#eef0f4;overflow:hidden;}
.fill{display:block;height:100%;border-radius:999px;}
.riskpill{display:inline-flex;align-items:center;gap:7px;font-weight:800;font-size:14px;padding:8px 14px;border-radius:12px;}
.riskpill .dot,.badge .dot{width:8px;height:8px;border-radius:50%;display:inline-block;}
/* similar table */
.simscroll{overflow-x:auto;}
.simtable{width:100%;border-collapse:collapse;}
.simtable th{text-align:left;font-size:11px;letter-spacing:.05em;text-transform:uppercase;color:#94a3b8;font-weight:700;padding:8px 12px;border-bottom:1px solid var(--line);position:sticky;top:0;background:#fff;z-index:2;}
.simtable td{padding:12px;border-bottom:1px solid #f4f5f9;font-size:14px;color:var(--ink);}
.simtable tr:last-child td{border-bottom:none;}
.t-title{font-weight:600;max-width:240px;}
.badge{display:inline-flex;align-items:center;gap:6px;font-weight:700;font-size:12px;padding:3px 10px;border-radius:999px;}
.simwrap{display:flex;align-items:center;gap:8px;}
.simtrack{width:92px;height:7px;border-radius:999px;background:#eef0f4;overflow:hidden;}
.simfill{display:block;height:100%;border-radius:999px;background:linear-gradient(90deg,#6d5efc,#8b7bff);}
.simpct{font-size:13px;font-weight:700;color:#475569;}
.muted{color:#94a3b8;}
/* exploration view */
.mgrid{display:grid;grid-template-columns:repeat(2,1fr);gap:14px;}
.mcard{border:1px solid var(--line);border-radius:14px;overflow:hidden;background:#fff;}
.mh{background:#0f172a;color:#fff;padding:11px 14px;font-weight:700;font-size:14px;display:flex;justify-content:space-between;align-items:center;}
.chip{background:rgba(255,255,255,.16);border-radius:999px;padding:1px 9px;font-size:11px;font-weight:700;}
.mb{padding:13px 15px;font-size:13px;color:var(--ink);line-height:1.5;}
.spec{margin-bottom:6px;} .spec>span{display:inline-block;width:84px;color:var(--muted);font-weight:700;vertical-align:top;}
.cav{color:#92400e;font-size:12px;margin-top:8px;padding-top:8px;border-top:1px dashed var(--line);}
.rescard{border:1px solid var(--line);border-radius:14px;padding:16px 18px;background:#fff;}
.rescard .sum{font-style:italic;color:#334155;margin:0 0 12px;} .note{color:#94a3b8;font-size:12px;}
.tag{padding:2px 9px;border-radius:999px;font-size:12px;font-weight:700;} .tag.ok{color:#047857;background:#ecfdf5;} .tag.warn{color:#b45309;background:#fffbeb;}
table.insp{width:100%;border-collapse:collapse;} table.insp td{padding:7px 6px;border-bottom:1px solid #f1f5f9;font-size:14px;} table.insp .k{color:var(--muted);width:120px;}
table.cm{border-collapse:collapse;font-size:12px;} table.cm th{color:var(--muted);padding:4px 7px;} table.cm td{padding:6px 9px;text-align:center;border:1px solid #f1f5f9;min-width:34px;}
table.rep{width:100%;border-collapse:collapse;font-size:13px;} table.rep th{text-align:left;color:var(--muted);border-bottom:2px solid var(--line);padding:5px;} table.rep td{padding:5px;border-bottom:1px solid #f1f5f9;}
.small{font-size:12px;margin-bottom:6px;}
.empty{color:#94a3b8;font-size:14px;padding:26px 6px;text-align:center;line-height:1.5;}
"""

with gr.Blocks(title="Triage Copilot", css=CSS, theme=gr.themes.Soft(primary_hue="indigo")) as demo:
    with gr.Row(elem_id="shell"):
        # ---------- sidebar (brand + clickable nav + profile) ----------
        with gr.Column(elem_id="sidebar", elem_classes="sb"):
            gr.HTML(BRAND_HTML)
            nav_triage   = gr.Button("✨  Triage Copilot", elem_id="nav-triage")
            nav_settings = gr.Button("⚙️  Settings", elem_id="nav-settings")
            gr.HTML(PROFILE_HTML, elem_id="profile-wrap")
        with gr.Column(elem_id="content"):
            with gr.Row(elem_id="topbar"):
                gr.HTML(HEADER, elem_id="topbar-left")

            # ---------- dashboard ----------
            with gr.Column(visible=True) as dashboard:
                with gr.Row(equal_height=True, elem_id="maintop"):
                    # Case details — INPUT (wider)
                    with gr.Column(scale=13, elem_classes="panel", elem_id="panel-input"):
                        gr.HTML(CASE_HEAD)
                        with gr.Row():
                            gr.HTML('<div class="field-head">Case summary <span class="req">*</span></div>', elem_id="cs-head")
                            load_ex = gr.Button("\U0001F4C4  Load example", elem_id="loadex", scale=0, visible=False)
                            clear_btn = gr.Button("Clear", elem_id="clear-top", scale=0)
                        summary = gr.Textbox(value="", lines=3, show_label=False, container=False,
                                             placeholder="Describe the customer's issue…")
                        count = gr.HTML(INIT_COUNT)
                        with gr.Row():
                            channel = gr.Dropdown(CHANNELS, value="email", label="\U0001F517 Channel")
                            plan    = gr.Dropdown(PLANS, value="Standard", label="\U0001F451 Plan tier")
                        with gr.Row():
                            month  = gr.Dropdown(list(range(1, 13)), value=6, label="\U0001F4C5 Month")
                            tenure = gr.Number(value=12, label="⏱️ Tenure (months)")
                        agent_pri = gr.Dropdown(["(none)"] + PRIORITIES, value="(none)", label="\U0001F465 Agent's priority (optional)")
                        tags = gr.Textbox(value="", label="\U0001F3F7️ Tags (optional)",
                                          placeholder="xero;integration;error")
                        run_btn = gr.Button("➤  Run triage", elem_id="runbtn")
                    # Triage prediction — OUTPUT (narrower)
                    with gr.Column(scale=10, elem_classes="panel", elem_id="panel-output"):
                        gr.HTML(PRED_HEAD)
                        prediction = gr.HTML(INIT_PRED)
                # Similar cases — SIMILARITY (full width)
                with gr.Column(elem_classes="panel", elem_id="panel-sim"):
                    gr.HTML(SIM_HEAD)
                    similar = gr.HTML(INIT_SIM)

            # ---------- settings ----------
            with gr.Column(visible=False) as settings_view:
                with gr.Column(elem_classes="panel"):
                    gr.HTML(SETTINGS_HEAD)
                    admin_switch = gr.Checkbox(value=False, label="Admin mode")
                    gr.HTML('<div class="field-sub">Admin mode unlocks the model exploration view and the "Load example" shortcut.</div>')
                    view_models_btn = gr.Button("\U0001F4D6  View models", elem_id="view-btn", visible=False)

            # ---------- exploration ----------
            with gr.Column(visible=False) as explore:
                back_btn = gr.Button("← Back to triage", elem_id="loadex")
                gr.Markdown("### Model cards")
                gr.HTML(model_cards())
                gr.Markdown("### Normalised held-out metrics")
                gr.Dataframe(metrics_table(), interactive=False)
                gr.Markdown("### Inspect a held-out case — predicted vs actual")
                _default_model = EXPLORE_ORDER[0]
                with gr.Row():
                    model_pick = gr.Dropdown([(EXPLORE_LABELS[k], k) for k in EXPLORE_ORDER],
                                             value=_default_model, label="Model")
                    case_pick = gr.Dropdown(case_choices(_default_model), value=0, label="Held-out case")
                inspect_out = gr.HTML(inspect(_default_model, 0))

                def set_cases(model_key):
                    """Repopulate the case dropdown for the chosen model and select the first case."""
                    choices = case_choices(model_key) if model_key else []
                    first = choices[0][1] if choices else None
                    return gr.update(choices=choices, value=first)

                model_pick.change(set_cases, model_pick, case_pick).then(
                    inspect, [model_pick, case_pick], inspect_out)
                case_pick.change(inspect, [model_pick, case_pick], inspect_out)
                gr.Markdown("### Confusion matrix & per-class report")
                model_pick2 = gr.Dropdown([(EXPLORE_LABELS[k], k) for k in EXPLORE_ORDER], value="m1_case_type", label="Model")
                with gr.Row():
                    cm_out = gr.HTML(confusion_html("m1_case_type"))
                    rep_out = gr.HTML(report_html("m1_case_type"))
                model_pick2.change(lambda k: (confusion_html(k), report_html(k)), model_pick2, [cm_out, rep_out])
                gr.Markdown("### Browse held-out cases")
                with gr.Row():
                    browse_pick = gr.Dropdown([(EXPLORE_LABELS[k], k) for k in EXPLORE_ORDER],
                                              value="m1_case_type", label="Model")
                    browse_sort = gr.Dropdown(["Case order", "Correct first", "Wrong first"],
                                              value="Case order", label="Sort by result")
                browse_out = gr.HTML(heldout_table("m1_case_type", "Case order"))
                browse_pick.change(heldout_table, [browse_pick, browse_sort], browse_out)
                browse_sort.change(heldout_table, [browse_pick, browse_sort], browse_out)
                gr.Markdown("### Training notebooks (the real notebooks used to build each model)")
                with gr.Row():
                    for k in DOWNLOAD_KEYS:
                        gr.DownloadButton(f"⬇ {k}", value=NOTEBOOK_PATH[k], size="sm")

    # ---- wiring ----
    inputs  = [summary, channel, plan, month, tenure, agent_pri, tags]
    outputs = [prediction, similar, tags]
    run_btn.click(triage, inputs, outputs)
    load_ex.click(pick_example, None, inputs).then(triage, inputs, outputs).then(counter, summary, count)
    clear_btn.click(clear_all, None, inputs + [prediction, similar, count])
    summary.change(counter, summary, count)

    # view switching: [dashboard, settings, explore]
    views = [dashboard, settings_view, explore]
    nav_triage.click(lambda: (gr.update(visible=True), gr.update(visible=False), gr.update(visible=False)), None, views)
    nav_settings.click(lambda: (gr.update(visible=False), gr.update(visible=True), gr.update(visible=False)), None, views)
    view_models_btn.click(lambda: (gr.update(visible=False), gr.update(visible=False), gr.update(visible=True)), None, views)
    back_btn.click(lambda: (gr.update(visible=True), gr.update(visible=False), gr.update(visible=False)), None, views)
    # admin mode reveals both the model view button and the Load example shortcut
    admin_switch.change(lambda on: (gr.update(visible=bool(on)), gr.update(visible=bool(on))),
                        admin_switch, [view_models_btn, load_ex])

print("dashboard app built | models:", len(MODEL))

## 6. Launch

In [ ]:
# demo.launch(server_name="127.0.0.1", server_port=7866, inbrowser=False)